In [78]:
import numpy as np
import pandas as pd

from MLstatkit import Delong_test
from model_evaluation import leave_one_patient_out_logistic_regression as reg, delta_model
from ar_model_utils import apply_sliding_window
from data_utils import *

from tqdm import tqdm
from joblib import Parallel, delayed
import os 
import warnings
warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [56]:
df = pd.read_parquet('df_for_paper_processed.parq')
df.head(5)

,pt_id,time_bin_time,days_since_dbs,lfp_left_raw,stim_left,lfp_right_raw,stim_right,lead_location,left_lead_model,right_lead_model,...,left_R2_rolling_avg_14d,left_R2_rolling_avg_21d,left_R2_rolling_avg_28d,left_delta_R2_rolling_avg_1d,left_delta_R2_rolling_avg_3d,left_delta_R2_rolling_avg_5d,left_delta_R2_rolling_avg_7d,left_delta_R2_rolling_avg_14d,left_delta_R2_rolling_avg_21d,left_delta_R2_rolling_avg_28d
0,AA001,00:00:00,-8,253.0,0.0,366.0,0.0,VC/VS,LEAD_B33015,LEAD_B33015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AA001,00:10:00,-8,181.0,0.0,312.0,0.0,VC/VS,LEAD_B33015,LEAD_B33015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AA001,00:20:00,-8,198.0,0.0,306.0,0.0,VC/VS,LEAD_B33015,LEAD_B33015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AA001,00:30:00,-8,185.0,0.0,269.0,0.0,VC/VS,LEAD_B33015,LEAD_B33015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AA001,00:40:00,-8,215.0,0.0,314.0,0.0,VC/VS,LEAD_B33015,LEAD_B33015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## LinAR-1 Regression Stats Table 

In [73]:
# Compile right hem features
if os.path.exists('data/left_right_r2_features.parq'):
    daily_df = pd.read_parquet('data/left_right_r2_features.parq')

else:
    window_size = 3

    df.reset_index(drop=True, inplace=True)
    hem = 'right'

    pt_groups = df.groupby('pt_id', group_keys=False)
    ar_features = [f'lfp_{hem}_OvER_interpolate_z_scored_lag_1']
    tqdm.pandas(desc=f'Applying autoregressive model for {hem} hem')
    hem_results_df = pt_groups.progress_apply(
        lambda g: apply_sliding_window(
            g, ar_features, f'lfp_{hem}_OvER_interpolate_z_scored', window_size=window_size),
        include_groups=False
    )
    df = df.merge(
        hem_results_df,
        how='left',
        left_index=True,
        right_index=True
    )

    daily_df = df.groupby(by=['pt_id', 'days_since_dbs']).head(1)

    daily_df = generate_delta_avg_features(daily_df, [f'lfp_{hem}_OvER_interpolate_z_scored'], hem)
    daily_df.to_parquet('data/left_right_r2_features.parq')

daily_df.query('~pt_id.str.contains("AA").values', inplace=True)
daily_df.head(5)

,pt_id,time_bin_time,days_since_dbs,lfp_left_raw,stim_left,lfp_right_raw,stim_right,lead_location,left_lead_model,right_lead_model,...,right_R2_rolling_avg_14d,right_R2_rolling_avg_21d,right_R2_rolling_avg_28d,right_delta_R2_rolling_avg_1d,right_delta_R2_rolling_avg_3d,right_delta_R2_rolling_avg_5d,right_delta_R2_rolling_avg_7d,right_delta_R2_rolling_avg_14d,right_delta_R2_rolling_avg_21d,right_delta_R2_rolling_avg_28d
372,B001,00:00:00,-49,35010.0,0.0,29025.0,0.0,VC/VS,LEAD_3387,LEAD_3387,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
373,B001,00:00:00,-48,21184.0,0.0,9742.0,0.0,VC/VS,LEAD_3387,LEAD_3387,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
374,B001,00:00:00,-47,12474.0,0.0,6284.0,0.0,VC/VS,LEAD_3387,LEAD_3387,...,0.783994,0.783994,0.783994,0.313248,0.313248,0.313248,0.313248,0.313248,0.313248,0.313248
375,B001,00:00:00,-46,15010.0,0.0,3943.0,0.0,VC/VS,LEAD_3387,LEAD_3387,...,0.789918,0.789918,0.789918,0.325095,0.319171,0.319171,0.319171,0.319171,0.319171,0.319171
376,B001,00:00:00,-45,5546.0,0.0,2620.0,0.0,VC/VS,LEAD_3387,LEAD_3387,...,0.762795,0.762795,0.762795,0.237801,0.292048,0.292048,0.292048,0.292048,0.292048,0.292048


In [39]:
reg_results = pd.DataFrame(columns = ['Hemisphere', 'Feature', 'Label', 'AUROC', 'BA', 'TPR', 'TNR'])

for Hemisphere in ['left', 'right']:
    for Feature in ['daily', 'delta', 'avg']:
        match Feature:
            case 'daily':
                feature = [f'lfp_{Hemisphere}_OvER_interpolate_R2']
            case 'delta':
                feature = [f'delta_lfp_{Hemisphere}_OvER_interpolate_R2']
            case 'avg':
                feature = [f'{Hemisphere}_delta_R2_rolling_avg_14d']

        results_dict = reg(daily_df, feature)

        reg_results.loc[len(reg_results)] = [Hemisphere, Feature, 'true', results_dict[0]['AUC'], results_dict[0]['balanced_accuracy'], results_dict[0]['true_positive_rate'], results_dict[0]['true_negative_rate']]
        
        AUROC_dist = []
        BA_dist = []
        TPR_dist = []
        TNR_dist = []

        def run_shuffle(_):
            shuffle_results = reg(daily_df, feature, shuffle=True)
            return shuffle_results[0]['AUC'], shuffle_results[0]['balanced_accuracy'], shuffle_results[0]['true_positive_rate'], shuffle_results[0]['true_negative_rate']

        n = 10000
        n_jobs = -1  # uses all available cores
        results = Parallel(n_jobs=n_jobs)(
            delayed(run_shuffle)(i) for i in tqdm(range(n), desc=f'Shuffling labels for {Feature} model, {Hemisphere} hem')
        )

        AUROC_dist, BA_dist, TPR_dist, TNR_dist = zip(*results)

        reg_results.loc[len(reg_results)] = [Hemisphere, Feature, 'shuffled', np.mean(AUROC_dist), np.mean(BA_dist), np.mean(TPR_dist), np.mean(TNR_dist)]

        def calc_p(dist, metric):
            return (np.sum(np.array(dist) >= reg_results[(reg_results.Hemisphere == Hemisphere) & (reg_results.Feature == Feature) & (reg_results.Label == 'true')][metric].values[0]) + 1) / (len(dist) + 1)
        
        AUROC_p = calc_p(AUROC_dist, 'AUROC')
        BA_p = calc_p(BA_dist, 'BA')
        TPR_p = calc_p(TPR_dist, 'TPR')
        TNR_p = calc_p(TNR_dist, 'TNR')
        reg_results.loc[len(reg_results)] = [Hemisphere, Feature, 'p_val', AUROC_p, BA_p, TPR_p, TNR_p]

reg_results.to_excel('tables/all_reg_stats.xlsx')

Shuffling labels for avg model, right hem: 100%|██████████| 10000/10000 [08:01<00:00, 20.78it/s]


## Regression Stats for LinAR-1, LinAR-k, and LinAR-k zoned models including old and new data sets

In [ ]:
# Load LinAR-k and LinAR-k zoned data
ark_df = pd.read_parquet('df_w_ark.parq')
arkz_df = pd.read_pickle('data/ark_zoned_all_pts_r2.pkl')

# Compile delta and averaged features
reg_df = pd.merge(daily_df, ark_df[['pt_id', 'days_since_dbs', 'lfp_left_OvER_interpolate_R2_ark']].drop_duplicates(subset=['pt_id', 'days_since_dbs']), on=['days_since_dbs', 'pt_id'], how='left', suffixes=(None, None))
reg_df = pd.merge(reg_df, arkz_df, on=['days_since_dbs', 'pt_id'], how='left', suffixes=(None, '_arkz'))
reg_df.rename(columns={'r2': 'R2_arkz'}, inplace=True)

# Generate features for old data set and all data
OLD_PAPER_PTS = {'B001': (-48, 100), 'B002': (-6, 296), 'B004': (-9, 803), 'B005': (-44, 578), 'B006': (-13, 585), 'B007': (-13, -1), 'B008': (-19, 141), 'B009': (1106, 1224), 'B010': (-51, -29), 'U001': (-25, 43), 'U003': (-20, 14)}
old_data = []
for pt in list(OLD_PAPER_PTS.keys()):
    (start, end) = OLD_PAPER_PTS[pt]
    old_data.append(reg_df.query('pt_id == @pt and @start <= days_since_dbs <= @end'))

old_data_df = pd.concat(old_data)
old_data_df = generate_delta_avg_features(old_data_df, ['lfp_left_OvER_interpolate_R2', 'lfp_left_OvER_interpolate_R2_ark', 'R2_arkz'], 'left')

reg_df = generate_delta_avg_features(reg_df, ['lfp_left_OvER_interpolate_R2_ark', 'R2_arkz'], 'left')
reg_df.head(5)

,pt_id,time_bin_time,days_since_dbs,lfp_left_raw,stim_left,lfp_right_raw,stim_right,lead_location,left_lead_model,right_lead_model,...,left_arkz_rolling_avg_14d,left_arkz_rolling_avg_21d,left_arkz_rolling_avg_28d,left_delta_arkz_rolling_avg_1d,left_delta_arkz_rolling_avg_3d,left_delta_arkz_rolling_avg_5d,left_delta_arkz_rolling_avg_7d,left_delta_arkz_rolling_avg_14d,left_delta_arkz_rolling_avg_21d,left_delta_arkz_rolling_avg_28d
0,B001,00:00:00,-49,35010.0,0.0,29025.0,0.0,VC/VS,LEAD_3387,LEAD_3387,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,B001,00:00:00,-48,21184.0,0.0,9742.0,0.0,VC/VS,LEAD_3387,LEAD_3387,...,0.260541,0.260541,0.260541,-0.382962,-0.382962,-0.382962,-0.382962,-0.382962,-0.382962,-0.382962
2,B001,00:00:00,-47,12474.0,0.0,6284.0,0.0,VC/VS,LEAD_3387,LEAD_3387,...,0.466252,0.466252,0.466252,0.028461,-0.177250,-0.177250,-0.177250,-0.177250,-0.177250,-0.177250
3,B001,00:00:00,-46,15010.0,0.0,3943.0,0.0,VC/VS,LEAD_3387,LEAD_3387,...,0.561254,0.561254,0.561254,0.107755,-0.082249,-0.082249,-0.082249,-0.082249,-0.082249,-0.082249
4,B001,00:00:00,-45,5546.0,0.0,2620.0,0.0,VC/VS,LEAD_3387,LEAD_3387,...,0.596018,0.596018,0.596018,0.056808,0.064341,-0.047484,-0.047484,-0.047484,-0.047484,-0.047484


In [77]:
# Regression stats AR(1), AR(k), and AR(k)-zoned models for both old and new data

reg_results = pd.DataFrame(columns = ['Model', 'Feature', 'Data', 'AUROC', 'BA', 'TPR', 'TNR'])
model_map = {'LinAR-1': 'lfp_left_OvER_interpolate_R2', 'LinAR-k': 'lfp_left_OvER_interpolate_R2_ark', 'LinAR-k (zoned)': 'R2_arkz'}

for Model in ['LinAR-1', 'LinAR-k', 'LinAR-k (zoned)']: # AR-1, AR-k, AR-k zoned
    for Feature in ['daily', 'delta', 'avg']:
        match Feature:
            case 'daily':
                feature = [model_map[Model]]
            case 'delta':
                feature = [f'delta_{model_map[Model]}']
            case 'avg':
                feature = [f'left_delta_{model_map[Model].split('_')[-1]}_rolling_avg_14d']
        for Data in ['old', 'all']:
            fold_df = old_data_df if Data == 'old' else reg_df
            print(f'Running regression for {Model} with {Feature.upper()} feature for {Data.upper()} data...')
            _results = reg(fold_df, feature)

            reg_results.loc[len(reg_results)] = [Model, Feature.upper(), Data.upper(), _results[0]['AUC'], _results[0]['balanced_accuracy'], _results[0]['true_positive_rate'], _results[0]['true_negative_rate']]

reg_results.to_excel('tables/ar_reg_stats.xlsx')

Running regression for LinAR-1 with DAILY feature for OLD data...
Running regression for LinAR-1 with DAILY feature for ALL data...
Running regression for LinAR-1 with DELTA feature for OLD data...
Running regression for LinAR-1 with DELTA feature for ALL data...
Running regression for LinAR-1 with AVG feature for OLD data...
Running regression for LinAR-1 with AVG feature for ALL data...
Running regression for LinAR-k with DAILY feature for OLD data...
Running regression for LinAR-k with DAILY feature for ALL data...
Running regression for LinAR-k with DELTA feature for OLD data...
Running regression for LinAR-k with DELTA feature for ALL data...
Running regression for LinAR-k with AVG feature for OLD data...
Running regression for LinAR-k with AVG feature for ALL data...
Running regression for LinAR-k (zoned) with DAILY feature for OLD data...
Running regression for LinAR-k (zoned) with DAILY feature for ALL data...
Running regression for LinAR-k (zoned) with DELTA feature for OLD da

In [86]:
_results[1]['B001'].keys()

dict_keys(['model', 'confusion_matrix', 'weighted_confusion_matrix', 'y_true', 'y_pred', 'y_prob', 'auc', 'balanced_accuracy', 'raw_accuracy'])

## Delong's test to compare model performances

In [90]:
# Delong's results for AR(k) vs AR(k)-zoned and AR(k) vs AR(1) models for both old and new data
def concat_dict(dict, key):
    arr = []
    for pt in list(dict.keys()):
        try:
            arr.extend(dict[pt][key])
        except KeyError:
            continue
    return np.array(arr)

delong_results = pd.DataFrame(columns=['Feature', 'Other Model', 'p', 'z', 'LinAR-1 AUC', 'Other Model AUC'])
for Feature in ['daily', 'delta', 'avg']:

    for Model in ['LinAR-k', 'LinAR-k (zoned)']:
            match Feature:
                case 'daily':
                    ar1_feature = [model_map['LinAR-1']]
                    feature = [model_map[Model]]
                case 'delta':
                    ar1_feature = [f'delta_{model_map['LinAR-1']}']
                    feature = [f'delta_{model_map[Model]}']
                case 'avg':
                    ar1_feature = [f'left_delta_{model_map['LinAR-1'].split('_')[-1]}_rolling_avg_14d']
                    feature = [f'left_delta_{model_map[Model].split('_')[-1]}_rolling_avg_14d']

            print(f'Comparing regressions for {Model} and LinAR-1 with {Feature} feature ...')

            fold_df = reg_df.dropna(subset=[*ar1_feature, *feature])
            ar1_results = reg(fold_df, ar1_feature)
            y_true = concat_dict(ar1_results[1], 'y_true')
            ar1_probs = concat_dict(ar1_results[1], 'y_prob')

            ark_results = reg(fold_df, feature)
            ark_probs = concat_dict(ark_results[1], 'y_prob')
        
            z, p, auc_ar1, auc_ark = Delong_test(y_true, ar1_probs, ark_probs, return_ci=False, return_auc=True, verbose=0)

            delong_results.loc[len(delong_results)] = [Feature, Model, p, z, auc_ar1, auc_ark]

delong_results['AUC Diff'] = delong_results['LinAR-1 AUC'] - delong_results['Other Model AUC']
delong_results.to_excel('tables/ark_delong_test_results.xlsx')

Comparing regressions for LinAR-k and LinAR-1 with daily feature ...
Comparing regressions for LinAR-k (zoned) and LinAR-1 with daily feature ...
Comparing regressions for LinAR-k and LinAR-1 with delta feature ...
Comparing regressions for LinAR-k (zoned) and LinAR-1 with delta feature ...
Comparing regressions for LinAR-k and LinAR-1 with avg feature ...
Comparing regressions for LinAR-k (zoned) and LinAR-1 with avg feature ...
